# 12장 · 샘플링 — 다음 단어를 고르는 법 🎲

웹 챕터 **"샘플링"** 과 짝을 이루는 **마지막** 실습입니다.
출력층이 준 확률에서 실제로 단어를 고르는 방법(**greedy · temperature · top-k · top-p**)을
직접 계산하고 뽑아 봅니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요
from tutor import chat

chat('ch12')

## 1. 다음 단어 확률 (출력층 결과)

"오늘 날씨가 정말 ___" 후보들의 점수(logits)입니다. (웹 실험실과 같은 값)

In [1]:
import numpy as np
rng = np.random.default_rng(7)

VOCAB = ['좋다', '춥다', '덥다', '흐리다', '사과', '달린다']
logits = np.array([3.0, 1.5, 1.0, 0.2, -0.5, -1.0])

def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()

p = softmax(logits)
for w, pi in zip(VOCAB, p):
    print(f'{w:5s}: {pi*100:5.1f}%')

좋다   :  68.1%
춥다   :  15.2%
덥다   :   9.2%
흐리다  :   4.1%
사과   :   2.1%
달린다  :   1.2%


## 2. greedy vs 샘플링

- **greedy**: 항상 확률 1등 (결정적, 매번 같음)
- **샘플링**: 확률에 따라 무작위 (매번 다를 수 있음)

In [2]:
print('greedy →', VOCAB[int(np.argmax(p))], '(항상 같음)')
print('샘플링 5번:', [VOCAB[rng.choice(len(VOCAB), p=p)] for _ in range(5)])

greedy → 좋다 (항상 같음)
샘플링 5번: ['좋다', '덥다', '춥다', '좋다', '좋다']


## 3. temperature: 분포 모양 바꾸기

소프트맥스 전에 점수를 T로 나눕니다: $\text{softmax}(z / T)$.
T가 작으면 뾰족(안전), 크면 평평(다양).

In [3]:
def softmax_T(z, T):
    return softmax(z / T)

for T in [0.3, 1.0, 2.0]:
    pt = softmax_T(logits, T)
    top = VOCAB[int(np.argmax(pt))]
    print(f'T={T}: 1등 {top} {pt.max()*100:4.1f}% | 분포 {np.round(pt, 2)}')

T=0.3: 1등 좋다 99.2% | 분포 [0.99 0.01 0.   0.   0.   0.  ]
T=1.0: 1등 좋다 68.1% | 분포 [0.68 0.15 0.09 0.04 0.02 0.01]
T=2.0: 1등 좋다 41.7% | 분포 [0.42 0.2  0.15 0.1  0.07 0.06]


## 4. top-k: 상위 k개만 남기기

확률 상위 k개만 후보로 두고 나머지는 버린 뒤(0), 다시 정규화해서 뽑습니다.

In [4]:
def top_k(p, k):
    idx = np.argsort(p)[::-1][:k]      # 확률 큰 순 상위 k
    masked = np.zeros_like(p); masked[idx] = p[idx]
    return masked / masked.sum()

for k in [1, 3, 6]:
    pk = top_k(p, k)
    survivors = [VOCAB[i] for i in range(len(VOCAB)) if pk[i] > 0]
    print(f'top-{k}: 후보 {survivors}')

top-1: 후보 ['좋다']
top-3: 후보 ['좋다', '춥다', '덥다']
top-6: 후보 ['좋다', '춥다', '덥다', '흐리다', '사과', '달린다']


## 5. top-p (뉴클리어스): 누적 확률로 자르기

확률 높은 순으로 더해, 누적이 p 를 넘는 순간까지의 단어만 후보로 둡니다.
확신할 땐 적게, 애매할 땐 많이, **개수가 유동적**입니다.

In [5]:
def top_p(p, thresh):
    order = np.argsort(p)[::-1]
    cum = np.cumsum(p[order])
    cutoff = np.searchsorted(cum, thresh) + 1   # 누적이 thresh 넘을 때까지
    keep = order[:cutoff]
    masked = np.zeros_like(p); masked[keep] = p[keep]
    return masked / masked.sum(), [VOCAB[i] for i in keep]

for thr in [0.5, 0.9]:
    _, survivors = top_p(p, thr)
    print(f'top-p={thr}: 후보 {survivors}')

top-p=0.5: 후보 ['좋다']
top-p=0.9: 후보 ['좋다', '춥다', '덥다']


## 6. 직접 해보기 ✏️

temperature를 아주 낮추면(T=0.1) 거의 항상 1등이 뽑힙니다(=greedy). 확인해 보세요.

In [6]:
pt = softmax_T(logits, 0.1)
picks = [VOCAB[rng.choice(len(VOCAB), p=pt)] for _ in range(10)]
print('T=0.1 로 10번 뽑기:', picks)

assert pt.max() > 0.99, 'T가 낮으면 1등 확률이 거의 1이어야!'
assert all(w == '좋다' for w in picks), '거의 항상 1등이 나와야 합니다!'
print('✅ 낮은 temperature ≈ greedy (거의 항상 같은 단어)')

T=0.1 로 10번 뽑기: ['좋다', '좋다', '좋다', '좋다', '좋다', '좋다', '좋다', '좋다', '좋다', '좋다']
✅ 낮은 temperature ≈ greedy (거의 항상 같은 단어)


---
## 🎉 전 과정 완주: 트랜스포머 정복!

소프트맥스 한 조각에서 시작해 → 토큰화·임베딩·위치·어텐션·블록·출력층 → 학습 → **샘플링**까지,
**LLM의 처음부터 끝까지**를 손으로 만지며 이해했습니다.

🎯 **궁극 목표 달성**: LLM은 *"다음 단어를 확률로 고르는 기계"*, 그 핵심 부품이 **트랜스포머**.
이제 ChatGPT를 보면 스스로 설명할 수 있어요. 🙂 수고하셨습니다!